In [51]:
import numpy as np
import pandas as pd
import os 
from pathlib import Path
print("현재 위치:", os.getcwd())

현재 위치: /Users/yuri.seo/Documents/GitHub/webnovel-pass-ab-test/notebooks


In [52]:
#데이터 불러오기

ROOT = Path.cwd().parent
DATA = ROOT / "data" / "clean"


coin_transactions = pd.read_csv(DATA / "coin_transactions.csv.gz",parse_dates=['txn_at'])
daily_reading = pd.read_csv(DATA / "daily_reading.csv.gz",parse_dates = ['read_date'])
exposures = pd.read_csv(DATA / "exposures.csv.gz",parse_dates = ['exposed_at'])
subscription_events = pd.read_csv(DATA / "subscription_events.csv.gz",parse_dates = ['event_at'])
titles = pd.read_csv(DATA / "titles.csv.gz")
users = pd.read_csv(DATA / "users.csv.gz",parse_dates = ['signup_date'])



# 각 테이블 정보 확인을 위해 딕셔너리 생성
table_info = {"coin_transactions":coin_transactions,
      "daily_reading":daily_reading,
     "exposures":exposures,
     "subscription_events":subscription_events,
     "titles":titles,
     "users":users}

for key, val in table_info.items():
    print(key, val.shape)


coin_transactions (823238, 6)
daily_reading (1859866, 6)
exposures (314922, 4)
subscription_events (60367, 5)
titles (6300, 4)
users (314922, 9)


# 전환율 확인

In [69]:
# variant별 전환율 구하기 

# base 테이블 만들기
# 전체 유저 | variant | started_trial | 결제 여부 (converted)

base = table_info["users"][["user_id"]]

base = base.merge(
    table_info["exposures"][["user_id","variant"]],
    on = "user_id",
    how = "left"
)


# base에 trial True =1 False = 0 으로 추가하기. 

# events로 먼저 정리 
events = table_info["subscription_events"].copy()

events["start_trial"] = (events["event_type"] == 'trial_start').astype(int)
events["converted"] = (events["event_type"] == 'payment').astype(int)
events

# user별 그룹화 
user_events = events.groupby("user_id")[["start_trial","converted"]].max().reset_index()

# Join
base = base.merge(
    user_events[["user_id","start_trial","converted"]],
    on = "user_id",
    how = "left"
)

# 결측치 0으로 변환 
base[["start_trial","converted"]] = base[["start_trial","converted"]].fillna(0).astype(int)

len(base)


# 각 그룹별 전환율 구하기
result_st = base.groupby("variant")["start_trial"].mean()                                       ## 체험율
result_st_convert = base.loc[base["start_trial"]==1].groupby("variant")["converted"].mean()     ## 체험시작 기준 전환율 (start_trial)
result_e_convert = base.groupby("variant")["converted"].mean()                                  ## 노출자 기준 전환율 (exposed)
result_user_cnt = base.groupby("variant")["user_id"].nunique()                                ## 전체 유저 수
result_st_cnt = base.groupby("variant")["start_trial"].sum()                                    ## 체험 유저 수
result_convert_cnt = base.groupby("variant")["converted"].sum()                                 ## 구매 유저 수


result_d = pd.DataFrame({
    "노출 유저 수" : result_user_cnt,
    "체험 유저 수" : result_st_cnt,
    "구매 유저 수" : result_convert_cnt,
    "체험 시작률"  : result_st,
    "노출 기준 전환율" : result_e_convert,
    "체험 시작 기준 전환율" : result_st_convert
})
result_d = result_d.T
result_d


variant,A,B
노출 유저 수,157461.000000,157461.000000
체험 유저 수,12323.000000,13805.000000
구매 유저 수,3602.000000,3977.000000
체험 시작률,0.078261,0.087673
노출 기준 전환율,0.022876,0.025257
체험 시작 기준 전환율,0.292299,0.288084


In [70]:
# ── 비율 행만 %로 바꾸기 ──
rate_rows = ["체험 시작률", "노출 기준 전환율", "체험 시작 기준 전환율"]
count_rows = ["노출 유저 수", "체험 유저 수", "구매 유저 수"]

result_d.loc[rate_rows] = result_d.loc[rate_rows] * 100

# ── 차이와 상대 uplift ──
# 절대 차이: B - A  (비율 행은 %p, 인원 행은 명)
result_d["차이(B-A)"] = result_d["B"] - result_d["A"]

# 상대 uplift: (B - A) / A × 100  → 비율 행에만 의미가 있음
result_d["상대 uplift(%)"] = (result_d["B"] - result_d["A"]) / result_d["A"] * 100
result_d.loc[count_rows, "상대 uplift(%)"] = np.nan   # 인원 수의 uplift는 의미 없으니 비움

# ── 보기 좋게 반올림 ──
result_d = result_d.round(2)
result_d

variant,A,B,차이(B-A),상대 uplift(%)
노출 유저 수,157461.00,157461.00,0.00,NaN
체험 유저 수,12323.00,13805.00,1482.00,NaN
구매 유저 수,3602.00,3977.00,375.00,NaN
체험 시작률,7.83,8.77,0.94,12.03
노출 기준 전환율,2.29,2.53,0.24,10.41
체험 시작 기준 전환율,29.23,28.81,-0.42,-1.44


### 라운드 1 결과: 첫인상

**관찰**
- 체험 시작률: A 7.83% vs B 8.77% → B가 0.94%p 높음 (상대 +12.0%)
- 노출 기준 전환율: A 2.29% vs B 2.53% → B가 0.24%p 높음 (상대 +10.4%)
- 체험 시작 기준 전환율: A 29.23% vs B 28.81% → A가 0.42%p 높음 (상대 -1.4%)

**첫인상**
- 어떤 전환율을 보느냐에 따라 우세한 그룹이 바뀜
- 노출 기준으로는 B가 나아 보이고, 체험 시작 기준으로는 A가 근소하게 나아 보임
- 세 지표 중 차이가 가장 큰 것은 체험 시작률 (상대 +12.0%)

**기준마다 결과가 다른 이유 (가설)**
- 노출 기준 전환율 = 체험 시작률 × 체험 시작 기준 전환율
- B는 "14일 무료" 문구로 체험을 더 많이 끌어왔고(체험 시작률 ↑), 그중 결제로 이어진 비율은 A와 비슷하거나 조금 낮음
- 그 결과 노출 기준으로는 B가 앞서고, 체험 시작 기준으로는 A가 앞섬

**남은 의문 → 라운드 2·3에서 확인**
1. 차이가 모두 1%p 미만. 우연히 생긴 차이일 가능성은? → 통계 검정 필요
2. 노출 유저 수가 A, B 정확히 같음. 배정이 제대로 이뤄졌나? → SRM 점검
3. 두 그룹은 실험 전에 비슷한 사람들이었나? → 사전 균형 점검
4. B가 체험을 더 많이 끌어왔다면, B의 체험 시작자에는 A라면 체험하지 않았을 사람이 섞여 있음. A·B의 체험 시작자를 그대로 비교해도 공정한가? → 선택 편향 점검
5. 출시 판단에는 어떤 전환율을 써야 하나? 각 지표는 어떤 질문에 답하는 숫자인가?

## 라운드 2. 실험 점검: 이 실험을 믿을 수 있는가

### 왜 점검부터 하는가
- 라운드 1에서 본 차이를 "B(14일 체험) 때문"이라고 말하려면, 두 그룹이 체험 기간 외에는 같은 조건이어야 함
- 이 전제가 깨져 있으면 검정을 아무리 정교하게 해도 결론을 신뢰할 수 없음
- 따라서 검정(라운드 3) 전에 아래 두 가지를 먼저 확인

| 점검 | 질문 | 실패 시 |
| --- | --- | --- |
| SRM | 계획한 비율(50:50)대로 배정됐는가 | 배정 시스템·로그 오류 의심 → 결과 전체 보류 |
| 사전 균형 | 실험 전 행동이 두 그룹에서 비슷한가 | 처음부터 다른 집단 → 차이의 원인을 B로 볼 수 없음 |

---

### 2-1. SRM (Sample Ratio Mismatch)

**질문**
- 50:50 배정이었다면 현재 관찰된 인원 비율이 나올 만한가

**고민: A·B 인원이 정확히 같은데 검정이 필요한가**
- 점검은 숫자를 보고 할지 말지 정하는 것이 아니라 항상 같은 절차로 수행
- 다른 실험에서 157,461 vs 156,890처럼 어긋났을 때 "이 정도면 괜찮은가"를 판단하는 기준이 검정
- 정확히 동일한 인원은 무작위 배정에서 흔하지 않음 (15만 명씩 나누면 보통 수백 명 차이 발생)
  - 실무라면 강제 균등 배정 방식인지 확인할 대상
  - SRM은 비율이 어긋난 것을 찾는 검정이므로 이 경우 실패 사유는 아님
  - 본 데이터에서는 생성 과정에서 우연히 일치한 것으로 판단

**후보 방법**

| 방법 | 내용 | 판단 |
| --- | --- | --- |
| 카이제곱 적합도 검정 | 관찰 인원과 기대 인원(전체의 50%씩) 비교 | ✅ 채택. SRM 점검의 표준 방법, 그룹이 3개 이상이어도 동일하게 적용 가능 |
| 이항 검정 | "A일 확률 0.5"인 시행 결과로 간주 | △ 두 그룹에서는 결과가 사실상 동일. 확장성 때문에 카이제곱 선택 |
| 비율 눈으로 확인 | 50.1% vs 49.9%면 괜찮다고 판단 | ✗ 판단 기준이 없음. 31만 명 규모에서는 0.3%p 차이도 이상 신호일 수 있음 |

**판단 기준: p < 0.001이면 SRM**
- 일반적인 0.05보다 엄격하게 설정 (실험 설계서 7장)
- SRM은 "실험 전체를 버린다"는 무거운 결정 → 우연한 흔들림으로 실험을 버리는 손실이 큼
- 실제 배정 오류는 보통 수천 명 단위로 어긋나므로 엄격한 기준으로도 충분히 탐지됨

---

### 2-2. 사전 균형 (Pre-experiment balance)

**질문**
- 실험 전에 이미 정해진 값들이 두 그룹에서 비슷한가
- 실험 전 값은 B의 영향을 받을 수 없으므로, 여기서 차이가 크면 배정 자체가 치우쳤다는 신호

**비교 대상**

| 유형 | 컬럼 |
| --- | --- |
| 숫자 | pre30_serial_reads, pre30_completed_reads, pre30_free_reads, pre30_coin_spend |
| 범주 | age_group, acquisition_channel, preferred_genre, platform |

**고민: t-검정으로 평균을 비교하면 되지 않나**
- 표본이 약 31만 명으로 매우 큼
- 표본이 크면 실질적으로 무의미한 작은 차이도 p-value가 0.05 아래로 떨어짐
  - 예: 실험 전 30일 열람이 A 45.1화, B 45.3화여도 "유의한 차이"로 나올 수 있음
  - 그렇다고 두 그룹이 다른 집단이라고 보기는 어려움
- 따라서 "차이가 있는가(p-value)"보다 "차이가 얼마나 큰가(효과 크기)"로 판단

**후보 방법**

| 방법 | 묻는 것 | 판단 |
| --- | --- | --- |
| t-검정 p-value | 차이가 0인가 | △ 표본이 크면 사소한 차이도 유의하게 나옴. 참고용 |
| 표준화 평균 차이 (SMD) | 차이가 분포 폭에 비해 얼마나 큰가 | ✅ 숫자 컬럼에 채택 |
| 그룹별 비율 비교 + 카이제곱 독립성 검정 | 범주 분포가 그룹마다 다른가 | ✅ 범주 컬럼에 채택. 비율 차이를 함께 확인 |

**SMD (Standardized Mean Difference)**
- SMD = (B 평균 − A 평균) ÷ 합친 표준편차
- 합친 표준편차 = √((A 분산 + B 분산) / 2)
- 단위가 사라져 열람 회차·결제 금액처럼 단위가 다른 컬럼을 같은 기준으로 비교 가능
- 판단 기준: |SMD| < 0.1이면 균형

**범주 컬럼 확인 방법**
- 그룹별 비율: crosstab(행 = variant, normalize = "index")로 그룹마다 합이 1이 되도록 구성
- 검정: 비율이 아닌 인원수 교차표로 카이제곱 독립성 검정
- 표본이 큰 만큼 p-value보다 비율 차이(%p) 크기를 함께 보고 판단

---

### 체크포인트
- [ ] SRM 카이제곱 p ≥ 0.001
- [ ] 숫자 컬럼 4개 |SMD| < 0.1
- [ ] 범주 컬럼 그룹 간 비율 차이 확인